In [12]:
import os
import time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
import segmentation_models_pytorch as smp
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


# ============= CORRECTED PATHS FOR YOUR SYSTEM =============
BASE_DIR = "/home/s202312054/project file "  # Full absolute path with trailing space
IMG_DIR = os.path.join(BASE_DIR, "image")     # lowercase "image" (not "Images")
MASK_DIR = os.path.join(BASE_DIR, "musk")     # "musk" (not "Masks")
CSV_PATH = os.path.join(BASE_DIR, "bus_data.csv")

CHECKPOINT_DIR = os.path.join(BASE_DIR, "MobileVnet3")
# ===========================================================

ENCODER_NAME = 'mobilenetv3_large_100'
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_EPOCHS = 500  # ← CHANGED from 50 to 500
LEARNING_RATE = 1e-4

TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10
RANDOM_SEED = 42


class BUSBRADataset(Dataset):
    def __init__(self, images_path, masks_path, size=(256, 256), transform=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.size = size
        self.transform = transform
    
    def __len__(self):
        return len(self.images_path)
    
    def __getitem__(self, index):
        image = cv2.imread(self.images_path[index], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks_path[index], cv2.IMREAD_GRAYSCALE)
        
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        
        image = cv2.resize(image, self.size)
        mask = cv2.resize(mask, self.size, interpolation=cv2.INTER_NEAREST)
        
        image = image.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)
        
        image = np.expand_dims(image, axis=0)
        mask = np.expand_dims(mask, axis=0)
        
        return torch.from_numpy(image), torch.from_numpy(mask)


def load_data(csv_path, img_dir, mask_dir):
    df = pd.read_csv(csv_path)
    unique_cases = df["Case"].unique()
    
    train_cases, temp_cases = train_test_split(
        unique_cases, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED
    )
    val_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_cases, test_cases = train_test_split(
        temp_cases, test_size=val_test_ratio, random_state=RANDOM_SEED
    )
    
    def get_paths(subset_df):
        img_paths, mask_paths = [], []
        for _, row in subset_df.iterrows():
            img_id = row["ID"]
            img_path = os.path.join(img_dir, img_id + ".png")
            mask_path = os.path.join(mask_dir, img_id.replace("bus_", "mask_") + ".png")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                img_paths.append(img_path)
                mask_paths.append(mask_path)
        return img_paths, mask_paths
    
    train_x, train_y = get_paths(df[df["Case"].isin(train_cases)])
    val_x, val_y = get_paths(df[df["Case"].isin(val_cases)])
    test_x, test_y = get_paths(df[df["Case"].isin(test_cases)])
    
    return (train_x, train_y), (val_x, val_y), (test_x, test_y)


class DiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
        return 1 - dice


class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.bce(pred, target)


def dice_coefficient(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
    return dice.item()


def iou_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + 1e-5) / (union + 1e-5)
    return iou.item()


def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    
    pbar = tqdm(loader, desc='Training')
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        
        dice = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{dice:.4f}'})
    
    return epoch_loss / len(loader), epoch_dice / len(loader)


def validate_epoch(model, loader, loss_fn, device):
    model.eval()
    epoch_loss = 0.0
    epoch_dice = 0.0
    epoch_iou = 0.0
    
    pbar = tqdm(loader, desc='Validation')
    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            
            dice = dice_coefficient(outputs, masks)
            iou = iou_score(outputs, masks)
            
            epoch_loss += loss.item()
            epoch_dice += dice
            epoch_iou += iou
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{dice:.4f}'})
    
    return epoch_loss / len(loader), epoch_dice / len(loader), epoch_iou / len(loader)


def plot_final_history(history, save_dir):
    """Plot final training history after all epochs complete"""
    epochs     = [h['epoch']      for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss   = [h['val_loss']   for h in history]
    train_dice = [h['train_dice'] for h in history]
    val_dice   = [h['val_dice']   for h in history]
    val_iou    = [h['val_iou']    for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'MobileNetV3-Large Training - Final Results (500 Epochs)', fontsize=14, fontweight='bold')

    # Loss plot
    axes[0].plot(epochs, train_loss, label='Train Loss', color='blue', linewidth=2)
    axes[0].plot(epochs, val_loss,   label='Val Loss',   color='orange', linewidth=2)
    axes[0].set_title('Loss vs Epochs', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=11)
    axes[0].set_ylabel('Loss', fontsize=11)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Dice plot
    axes[1].plot(epochs, train_dice, label='Train Dice', color='green', linewidth=2)
    axes[1].plot(epochs, val_dice,   label='Val Dice',   color='red', linewidth=2)
    axes[1].set_title('Dice Score vs Epochs', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=11)
    axes[1].set_ylabel('Dice Score', fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    # IoU plot
    axes[2].plot(epochs, val_iou, label='Val IoU', color='purple', linewidth=2)
    axes[2].set_title('IoU Score vs Epochs', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Epoch', fontsize=11)
    axes[2].set_ylabel('IoU Score', fontsize=11)
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'final_training_results.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n{'='*60}")
    print(f"📊 Final training graph saved: {save_path}")
    print(f"{'='*60}")


def main():
    print("=" * 60)
    print("BUS-BRA TRAINING - MobileNetV3-Large")
    print(f"Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE}")
    print(f"Split: Train {int(TRAIN_RATIO*100)}% | Val {int(VAL_RATIO*100)}% | Test {int(TEST_RATIO*100)}%")
    print("=" * 60)
    
    # ============= VERIFY PATHS =============
    print("\nVerifying paths...")
    print(f"BASE_DIR: {os.path.abspath(BASE_DIR)}")
    print(f"  Exists: {os.path.exists(BASE_DIR)}")
    print(f"CSV_PATH: {CSV_PATH}")
    print(f"  Exists: {os.path.exists(CSV_PATH)}")
    print(f"IMG_DIR: {IMG_DIR}")
    print(f"  Exists: {os.path.exists(IMG_DIR)}")
    print(f"MASK_DIR: {MASK_DIR}")
    print(f"  Exists: {os.path.exists(MASK_DIR)}")
    
    if not os.path.exists(BASE_DIR):
        raise FileNotFoundError(f"BASE_DIR not found: {BASE_DIR}")
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")
    if not os.path.exists(IMG_DIR):
        raise FileNotFoundError(f"IMG_DIR not found: {IMG_DIR}")
    if not os.path.exists(MASK_DIR):
        raise FileNotFoundError(f"MASK_DIR not found: {MASK_DIR}")
    
    print("✓ All paths verified!")
    # ========================================
    
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(RANDOM_SEED)
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✓ Checkpoint directory ready: {os.path.abspath(CHECKPOINT_DIR)}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    print("\nLoading data...")
    (train_x, train_y), (val_x, val_y), (test_x, test_y) = load_data(CSV_PATH, IMG_DIR, MASK_DIR)
    print(f"Train: {len(train_x)} | Val: {len(val_x)} | Test: {len(test_x)}")
    
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
        A.RandomBrightnessContrast(p=0.3),
    ])
    
    train_dataset = BUSBRADataset(train_x, train_y, IMAGE_SIZE, train_transform)
    val_dataset = BUSBRADataset(val_x, val_y, IMAGE_SIZE, None)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    print("\nBuilding model...")
    print(f"Encoder: {ENCODER_NAME}")
    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights='imagenet',
        in_channels=1,
        classes=1,
        activation=None
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    loss_fn = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_val_dice = 0.0
    best_epoch = 0
    
    history = []
    
    print("\nStarting training...")
    print(f"Total epochs: {NUM_EPOCHS}")
    print("=" * 60)
    
    for epoch in range(1, NUM_EPOCHS + 1):
        start_time = time.time()
        
        train_loss, train_dice = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_dice, val_iou = validate_epoch(model, val_loader, loss_fn, device)
        
        elapsed = time.time() - start_time
        mins = int(elapsed / 60)
        secs = int(elapsed - (mins * 60))
        
        print(f"\nEpoch [{epoch:03d}/{NUM_EPOCHS}] {mins}m {secs}s")
        print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")
        
        scheduler.step(val_loss)
        
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
            print(f"  ★ New best Val Dice: {val_dice:.4f}")
        
        history.append({
            'epoch':      epoch,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'train_dice': train_dice,
            'val_dice':   val_dice,
            'val_iou':    val_iou
        })

        # Save checkpoints every 50 epochs
        if epoch % 50 == 0:
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'model_epoch_{epoch}.pth'))
            print(f"  💾 Checkpoint saved: model_epoch_{epoch}.pth")
    
    # Save final model
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'final_model.pth'))
    
    # Plot ONLY the final graph after all training is complete
    plot_final_history(history, CHECKPOINT_DIR)

    print("\n" + "=" * 60)
    print("🎉 TRAINING COMPLETE!")
    print("=" * 60)
    print(f"\n📈 Best Val Dice: {best_val_dice:.4f} at epoch {best_epoch}")
    print(f"📁 Final model saved: {CHECKPOINT_DIR}/final_model.pth")
    print(f"⭐ Best model saved: {CHECKPOINT_DIR}/best_model.pth")
    
    # Save training history as JSON
    import json
    with open(os.path.join(CHECKPOINT_DIR, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    print(f"📊 Training history saved: {CHECKPOINT_DIR}/training_history.json")
    print(f"📂 All checkpoints saved to: {os.path.abspath(CHECKPOINT_DIR)}")


if __name__ == "__main__":
    main()

BUS-BRA TRAINING - MobileNetV3-Large
Epochs: 500 | Batch Size: 8
Split: Train 70% | Val 20% | Test 10%

Verifying paths...
BASE_DIR: /home/s202312054/project file 
  Exists: False
CSV_PATH: /home/s202312054/project file /bus_data.csv
  Exists: False
IMG_DIR: /home/s202312054/project file /image
  Exists: False
MASK_DIR: /home/s202312054/project file /musk
  Exists: False


FileNotFoundError: BASE_DIR not found: /home/s202312054/project file 